# SupplyPulse AI — Explainable AI (SHAP) & What-If Simulation

This notebook demonstrates local and global explainability using `shap.TreeExplainer` on the production XGBoost pipelines.

In [ ]:
import pandas as pd
import numpy as np
import shap
import joblib
import os
import matplotlib.pyplot as plt

print(f"SHAP version: {shap.__version__}")

## 1. Initialize TreeExplainer on Production Pipeline

In [ ]:
model_path = os.path.join("..", "models", "xgb_regressor.pkl")
if not os.path.exists(model_path):
    model_path = os.path.join("models", "xgb_regressor.pkl")

pipeline = joblib.load(model_path)
model = pipeline.named_steps["model"]
preprocessor = pipeline.named_steps["preprocessor"]

explainer = shap.TreeExplainer(model)
print(f"TreeExplainer Base Value E[f(x)]: {explainer.expected_value}")

## 2. Compute Local SHAP Attributions for a Sample Shipment

In [ ]:
data_path = os.path.join("..", "data", "processed", "test.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "processed", "test.csv")
df = pd.read_csv(data_path)

feature_cols = preprocessor.transformers_[0][2] + preprocessor.transformers_[1][2]
sample_df = df.iloc[[0]][feature_cols]
X_trans = preprocessor.transform(sample_df)

shap_vals = explainer.shap_values(X_trans)
pred_val = pipeline.predict(sample_df)[0]

print(f"Model Prediction: {pred_val:.2f}")
print(f"Base Expected Value: {explainer.expected_value[0]:.2f}")
print(f"Sum of SHAP Attributions: {shap_vals[0].sum():.2f}")
print(f"Verification (Base + Sum): {explainer.expected_value[0] + shap_vals[0].sum():.2f}")

## 3. Top Drivers Bar Plot

In [ ]:
shap_series = pd.Series(shap_vals[0], index=feature_cols).sort_values(key=abs, ascending=False).head(10)

plt.figure(figsize=(10, 6))
colors = ["#E76F51" if x < 0 else "#00A896" for x in shap_series.values]
shap_series.plot(kind="barh", color=colors)
plt.axvline(0, color="black", linestyle="--", linewidth=0.8)
plt.title("Top 10 Local Feature Attributions (SHAP)")
plt.xlabel("SHAP Value (Contribution to Health Score)")
plt.gca().invert_yaxis()
plt.show()